# 7장. Router Agent 사용 계층적 에이전트 호출 연결

## 터미널에서 2개의 ACP 서버 실행 중지
crt + C 단축키로 서버 실행 중지하기

## FastACP 파일 생성

In [1]:
%%writefile fastacp.py

# 필요한 라이브러리 및 모듈 임포트
from typing import List, Dict, Callable, Optional, Union, Any, AsyncGenerator
import importlib.resources
import yaml
import json
from dataclasses import dataclass
from enum import Enum
from colorama import Fore
from acp_sdk.client import Client
from acp_sdk.models import (
    Message,
    MessagePart,
)

# === AgentCollection 구현 ===

class Agent:
    """ACP 에이전트를 표현하는 클래스입니다."""
    
    def __init__(self, name: str, description: str, capabilities: List[str]):
        # 에이전트의 이름, 설명, 능력을 초기화합니다.
        self.name = name
        self.description = description
        self.capabilities = capabilities
    
    def __str__(self):
        # 에이전트의 문자열 표현을 반환합니다.
        return f"Agent(name='{self.name}', description='{self.description}')"


class AgentCollection:
    """
    ACP 서버에서 사용 가능한 에이전트들의 컬렉션입니다.
    사용자가 ACP 서버에서 사용 가능한 에이전트를 찾을 수 있도록 합니다.
    """
    
    def __init__(self):
        # 에이전트 목록을 초기화합니다.
        self.agents = []
    
    @classmethod
    async def from_acp(cls, *servers) -> 'AgentCollection':
        """
        제공된 ACP 서버에서 에이전트를 가져와 AgentCollection을 생성합니다.
        
        Args:
            *servers: 에이전트를 가져올 ACP 서버 클라이언트 인스턴스들
            
        Returns:
            AgentCollection: 발견된 모든 에이전트를 포함하는 컬렉션
        """
        collection = cls()
        
        # 각 서버를 순회하며 에이전트를 가져옵니다.
        for server in servers:
            async for agent in server.agents():
                collection.agents.append((server, agent))
        
        return collection
    
    def get_agent(self, name: str) -> Optional[Agent]:
        """
        컬렉션에서 이름으로 에이전트를 찾습니다.
        
        Args:
            name: 찾을 에이전트의 이름
            
        Returns:
            Agent or None: 찾은 에이전트 또는 찾지 못한 경우 None
        """
        for agent in self.agents:
            if agent.name == name:
                return agent
        return None
    
    def __iter__(self):
        """컬렉션의 모든 에이전트에 대해 반복할 수 있도록 합니다."""
        return iter(self.agents)


# === ACPCallingAgent 구현 ===

@dataclass
class ToolCall:
    """이름, 인수, 선택적 ID를 가진 도구 호출을 나타냅니다."""
    name: str
    arguments: Union[Dict[str, Any], str]
    id: Optional[str] = None


@dataclass
class ChatMessage:
    """내용과 선택적 도구 호출을 가진 채팅 메시지를 나타냅니다."""
    content: Optional[str]
    tool_calls: Optional[List[ToolCall]] = None
    raw: Any = None


class LogLevel(Enum):
    """로그 레벨을 정의하는 열거형입니다."""
    DEBUG = "debug"
    INFO = "info"
    WARNING = "warning"
    ERROR = "error"


class Logger:
    """에이전트 작업을 위한 간단한 로거입니다."""
    
    def log(self, content, level=LogLevel.INFO):
        # 로그 메시지를 콘솔에 출력합니다.
        print(f"[{level.value.upper()}] {content}")
    
    def log_markdown(self, content, title=None, level=LogLevel.INFO):
        # 마크다운 형식의 로그를 출력합니다.
        if title:
            print(f"[{level.value.upper()}] {title}")
        print(f"[{level.value.upper()}] {content}")


class AgentError(Exception):
    """에이전트 오류를 위한 기본 클래스입니다."""
    
    def __init__(self, message, logger=None):
        super().__init__(message)
        if logger:
            logger.log(message, level=LogLevel.ERROR)


class AgentParsingError(AgentError):
    """에이전트 출력을 파싱할 때 발생하는 오류입니다."""
    pass


class AgentToolCallError(AgentError):
    """도구를 호출할 때 발생하는 오류입니다."""
    pass


class AgentToolExecutionError(AgentError):
    """도구를 실행할 때 발생하는 오류입니다."""
    pass


class ActionStep:
    """에이전트의 추론 과정에서의 한 단계를 나타냅니다."""
    
    def __init__(self):
        self.model_input_messages = []
        self.model_output_message = None
        self.model_output = None
        self.tool_calls = []
        self.action_output = None
        self.observations = None


class Tool:
    """에이전트가 사용할 수 있는 도구의 기본 클래스입니다."""
    
    def __init__(self, name, description, inputs, output_type, client=None):
        self.name = name
        self.description = description
        self.inputs = inputs
        self.output_type = output_type
        self.client = client
    
    async def __call__(self, *args, **kwargs):
        # 도구가 호출될 때 실행되는 로직입니다.
        print(Fore.YELLOW + 'Tool being called with args: ' + str(args) + ' and kwargs: ' + str(kwargs) + Fore.RESET)
    
        # args 또는 kwargs에서 입력 내용을 추출합니다.
        content = ""
        if args and isinstance(args[0], str):
            content = args[0]
        elif "prompt" in kwargs:
            content = kwargs["prompt"]
        elif "input" in kwargs:
            content = kwargs["input"]
        elif kwargs:
            # 특정 키를 찾지 못하면 첫 번째 값을 사용합니다.
            content = next(iter(kwargs.values()))
            
        # 추출된 내용을 사용하여 메시지를 보냅니다.
        print(Fore.MAGENTA + content + Fore.RESET) 
        response = await self.client.run_sync(
            agent=self.name, 
            input=[Message(parts=[MessagePart(content=content, content_type="text/plain")])]
        )
        print(Fore.RED + str(response) + Fore.RESET) 
        return response.output[0].parts[0].content


class MultiStepAgent:
    """여러 단계로 작동하는 에이전트의 기본 클래스입니다."""
    
    def __init__(
        self,
        tools: Dict[str, Tool],
        model: Callable,
        prompt_templates: Dict[str, str],
        planning_interval: Optional[int] = None,
        **kwargs
    ):
        self.tools = tools
        self.model = model
        self.prompt_templates = prompt_templates
        self.planning_interval = planning_interval
        self.managed_agents = kwargs.get("managed_agents", {})
        self.logger = Logger()
        self.state = {}
        self.input_messages = []
    
    def initialize_system_prompt(self) -> str:
        """에이전트를 위한 시스템 프롬프트를 생성합니다."""
        raise NotImplementedError
    
    def write_memory_to_messages(self):
        """에이전트 메모리를 모델을 위한 메시지 목록으로 변환합니다."""
        # 구현은 메모리 구조에 따라 달라집니다.
        return self.input_messages
    
    async def step(self, memory_step: ActionStep) -> Union[None, Any]:
        """에이전트의 추론 과정에서 한 단계를 수행합니다."""
        raise NotImplementedError


def populate_template(template: str, variables: Dict[str, Any]) -> str:
    """변수로 템플릿을 채우는 헬퍼 함수입니다."""
    result = template
    for key, value in variables.items():
        placeholder = "{" + key + "}"
        result = result.replace(placeholder, str(value))
    return result


class ACPCallingAgent(MultiStepAgent):
    """
    이 에이전트는 ToolCallingAgent가 로컬 도구를 사용하는 것과 유사하게,
    원격 ACP 에이전트를 대상으로 JSON과 유사한 ACP 에이전트 호출을 사용합니다.
    
    Args:
        acp_agents (`dict[str, Agent]`): 이 에이전트가 호출할 수 있는 ACP 에이전트들입니다.
        model (`Callable[[list[dict[str, str]]], ChatMessage]`): 에이전트의 행동을 생성할 모델입니다.
        prompt_templates ([`Dict[str, str]`], *optional*): 프롬프트 템플릿입니다.
        planning_interval (`int`, *optional*): 에이전트가 계획 단계를 실행할 간격입니다.
        **kwargs: 추가 키워드 인수입니다.
    """
    
    def __init__(
        self,
        acp_agents: Dict[str, Agent],
        model: Callable[[List[Dict[str, str]]], ChatMessage],
        prompt_templates: Optional[Dict[str, str]] = None,
        planning_interval: Optional[int] = None,
        **kwargs,
    ):
        # 제공된 프롬프트 템플릿이 없는 경우 기본값을 사용합니다.
        if prompt_templates is None:
            prompt_templates = {
                # --- [ 한글 번역 프롬프트 ] ---
                "system_prompt": """당신은 전문 ACP 에이전트에게 작업을 위임할 수 있는 감독 에이전트입니다.
                사용 가능한 에이전트:
                {agents}

                당신의 임무는 다음과 같습니다:
                1. 사용자의 요청을 분석합니다.
                2. 정보를 수집하기 위해 적절한 에이전트를 호출합니다.
                3. 완전한 답변이 준비되면, 항상 final_answer 도구를 호출하여 응답을 제출합니다.
                4. 메시지에서 직접 답변을 제공하지 마세요 - 항상 final_answer 도구를 사용하세요.
                5. 작업을 완료하기에 충분한 정보가 있다면, 필요하지 않은 한 다른 에이전트를 호출하지 마세요.

                기억하세요:
                - 완전한 답변이 준비되면 항상 final_answer 도구를 사용하세요.
                - 일반 메시지에서는 답변을 제공하지 마세요.
                - 필요한 모든 정보를 수집하기 위해 여러 에이전트 호출을 연결하세요.
                - final_answer 도구는 사용자에게 결과를 반환하는 유일한 방법입니다.
                """
            }
        
        # ACP 에이전트를 도구와 유사한 형식으로 변환합니다.
        acp_tools = {}
        for name, agent in acp_agents.items():
            # ACP 에이전트를 호출할 호출 가능 객체(callable)를 생성합니다.
            acp_tools[name] = Tool(
                name=name,
                description=agent['agent'].description,
                inputs={"input": {"type":"string","description":"the prompt to pass to the agent"}},
                output_type="str",
                client=agent['client']
            )
            
            # 실제로 ACP 에이전트를 호출하도록 __call__ 메서드를 오버라이드합니다.
            def make_caller(agent_name, client):
                async def call_agent(prompt, **kwargs):
                    print(f"Calling {agent_name} with prompt: {prompt}")
                    response = await client.run_sync(
                        agent=agent_name, 
                        inputs=[Message(parts=[MessagePart(content=prompt, content_type="text/plain")])]
                    )
                    return response.outputs[0].parts[0].content
                return call_agent

            # 이 줄은 실제로는 에이전트의 클라이언트를 사용하지 않습니다. (코드의 잠재적 오류)
            acp_tools[name].__call__ = make_caller(name, agent['client'])
        
        # final_answer 도구를 추가합니다.
        acp_tools["final_answer"] = Tool(
            name="final_answer",
            description="Provide the final answer to the user's request",
            inputs={"answer": "The final answer to provide to the user"},
            output_type="str"
        )
        
        # final_answer 도구의 동작을 정의합니다.
        async def final_answer(answer, **kwargs):
            return answer
        
        acp_tools["final_answer"].__call__ = final_answer
        
        # 부모 클래스의 생성자를 호출합니다.
        super().__init__(
            tools=acp_tools,
            model=model,
            prompt_templates=prompt_templates,
            planning_interval=planning_interval,
            **kwargs,
        )
        
        self.acp_agents = acp_agents
    
    def initialize_system_prompt(self) -> str:
        """ACP 에이전트 정보로 시스템 프롬프트를 생성합니다."""
        agent_descriptions = "\n".join(
            [f"- {name}: {agent['agent'].description}" for name, agent in self.acp_agents.items()]
        )
        
        system_prompt = populate_template(
            self.prompt_templates["system_prompt"],
            variables={"agents": agent_descriptions},
        )
        return system_prompt

    def save_to_memory(self, key: str, value: Any) -> None:
        """에이전트의 영구 메모리에 값을 저장합니다."""
        self.state[key] = value
        self.logger.log(f"Saved to memory: {key}={value}", level=LogLevel.DEBUG)



    async def step(self, memory_step: ActionStep) -> Union[None, Any]:
        """
        추론 과정의 한 단계를 수행합니다: 에이전트가 생각하고, ACP 에이전트를 호출하고, 결과를 관찰합니다.
        단계가 최종이 아닌 경우 None을 반환합니다.
        """
        # 메시지를 LiteLLM 형식으로 변환합니다.
        memory_messages = self.write_memory_to_messages()
        # 모든 메시지가 올바른 LiteLLM 형식인지 확인합니다.
        for i, message in enumerate(memory_messages):
            if "content" in message and not isinstance(message["content"], list):
                memory_messages[i]["content"] = [{"type": "text", "text": message["content"]}]
        
        self.input_messages = memory_messages
        memory_step.model_input_messages = memory_messages.copy()

        try:  
            # 모델을 호출하여 다음 행동을 결정합니다.
            model_message: ChatMessage = self.model(
                memory_messages,
                tools_to_call_from=list(self.tools.values())[:-1],
                stop_sequences=["Observation:", "Calling agents:"],
            )           
            memory_step.model_output_message = model_message
        except Exception as e:
            raise AgentParsingError(f"Error while generating or parsing output:\n{e}", self.logger) from e
        
        # LLM의 출력 메시지를 로깅합니다.
        self.logger.log_markdown(
            content=model_message.content if model_message.content else str(model_message.raw),
            title="Output message of the LLM:",
            level=LogLevel.DEBUG,
        )
        
        # 모델이 도구/에이전트를 호출했는지 확인합니다.
        if not hasattr(model_message, 'tool_calls') or model_message.tool_calls is None or len(model_message.tool_calls) == 0:
            # 도구 호출이 없는 경우, 내용을 최종 답변으로 처리하려고 시도합니다.
            if model_message.content and "final_answer" in model_message.content.lower():
                self.logger.log(
                    f"Final answer detected in content: {model_message.content}",
                    level=LogLevel.INFO,
                )
                memory_step.action_output = model_message.content
                return model_message.content
            else:
                # 간단한 파서를 사용하여 내용에서 도구 호출을 추출하려고 시도합니다.
                content = model_message.content or ""
                if "tool:" in content.lower() or "agent:" in content.lower():
                    try:
                        # 매우 간단한 추출 - 정규식으로 향상될 수 있습니다.
                        lines = content.split('\n')
                        tool_line = next((line for line in lines if "tool:" in line.lower() or "agent:" in line.lower()), None)
                        if tool_line:
                            parts = tool_line.split(":", 1)
                            if len(parts) > 1:
                                agent_name = parts[1].strip()
                                # 인수를 찾습니다.
                                arg_line = next((line for line in lines if "arguments:" in line.lower()), None)
                                agent_arguments = {}
                                if arg_line:
                                    arg_parts = arg_line.split(":", 1)
                                    if len(arg_parts) > 1:
                                        try:
                                            # JSON으로 파싱을 시도합니다.
                                            agent_arguments = json.loads(arg_parts[1].strip())
                                        except json.JSONDecodeError:
                                            # 유효한 JSON이 아니면 문자열로 사용합니다.
                                            agent_arguments = {"prompt": arg_parts[1].strip()}
                                else:
                                    # 인수 라인이 없으면 나머지 내용을 프롬프트로 사용합니다.
                                    remaining_content = "\n".join(lines[lines.index(tool_line)+1:])
                                    agent_arguments = {"prompt": remaining_content.strip()}
                                
                                # 가상의 도구 호출을 생성합니다.
                                memory_step.model_output = str(f"Called Agent: '{agent_name}' with arguments: {agent_arguments}")
                                memory_step.tool_calls = [ToolCall(name=agent_name, arguments=agent_arguments, id="synthetic_id")]
                                
                                # 아래에서 추출된 도구 호출을 처리합니다.
                                return await self._process_tool_call(memory_step, agent_name, agent_arguments)
                    except Exception as e:
                        self.logger.log(f"Error parsing tool call from content: {e}", level=LogLevel.ERROR)
                
                raise AgentParsingError(
                    "Model did not call any agents and no final answer detected. Content: " + (model_message.content or "None"), 
                    self.logger
                )
        
        # 도구 호출을 처리합니다.
        tool_call = model_message.tool_calls[0]
        
        # 다양한 도구 호출 형식 구조를 처리합니다.
        if hasattr(tool_call, 'function') and hasattr(tool_call.function, 'name'):
            # 표준 OpenAI와 유사한 형식
            agent_name = tool_call.function.name
            agent_arguments = tool_call.function.arguments
            tool_call_id = getattr(tool_call, 'id', 'unknown_id')
        elif hasattr(tool_call, 'name'):
            # 단순화된 형식
            agent_name = tool_call.name
            agent_arguments = getattr(tool_call, 'arguments', {})
            tool_call_id = getattr(tool_call, 'id', 'unknown_id')
        else:
            # 딕셔너리로 파싱을 시도합니다.
            agent_name = tool_call.get('name', tool_call.get('function', {}).get('name', 'unknown'))
            agent_arguments = tool_call.get('arguments', tool_call.get('function', {}).get('arguments', {}))
            tool_call_id = tool_call.get('id', 'unknown_id')
            
        memory_step.model_output = str(f"Called Agent: '{agent_name}' with arguments: {agent_arguments}")
        memory_step.tool_calls = [ToolCall(name=agent_name, arguments=agent_arguments, id=tool_call_id)]
        
        return await self._process_tool_call(memory_step, agent_name, agent_arguments)
        
    async def _process_tool_call(self, memory_step: ActionStep, agent_name: str, agent_arguments: Any) -> Union[None, Any]:
        """
        주어진 이름과 인수로 도구 호출을 처리합니다.
        """
        
        # 도구 호출을 실행합니다.
        self.logger.log(
            f"Calling agent: '{agent_name}' with arguments: {agent_arguments}",
            level=LogLevel.INFO,
        )
        
        if agent_name == "final_answer":
            # 최종 답변을 처리합니다.
            if isinstance(agent_arguments, dict):
                if "answer" in agent_arguments:
                    answer = agent_arguments["answer"]
                else:
                    answer = agent_arguments
            else:
                answer = agent_arguments
            
            # 답변이 상태 변수를 참조하는 경우 해당 변수를 사용합니다.
            if isinstance(answer, str) and answer in self.state:
                final_answer = self.state[answer]
                self.logger.log(
                    f"Final answer: Extracting key '{answer}' from state to return value '{final_answer}'.",
                    level=LogLevel.INFO,
                )
            else:
                final_answer = answer
                self.logger.log(
                    f" {final_answer}",
                    level=LogLevel.INFO,
                )
            
            memory_step.action_output = final_answer
            return final_answer
        else:
            # ACP 에이전트 호출을 실행합니다.
            if agent_arguments is None:
                agent_arguments = {}
            
            observation = await self.execute_tool_call(agent_name, agent_arguments)
            updated_information = str(observation).strip()

            # 관찰 결과를 메모리에 저장합니다.
            self.save_to_memory(f"{agent_name}_response", updated_information)
            
            self.logger.log(
                f"Observations: {updated_information}",
                level=LogLevel.INFO,
            )
            
            memory_step.observations = updated_information
            return None
    
    def _substitute_state_variables(self, arguments: Union[Dict[str, str], str]) -> Union[Dict[str, Any], str]:
        """인수의 문자열 값을 해당하는 상태 값으로 대체합니다(존재하는 경우)."""
        if isinstance(arguments, dict):
            return {
                key: self.state.get(value, value) if isinstance(value, str) else value
                for key, value in arguments.items()
            }
        return arguments
    
    async def execute_tool_call(self, agent_name: str, arguments: Union[Dict[str, str], str]) -> Any:
        """
        제공된 인수로 ACP 에이전트 호출을 실행합니다.
        
        Args:
            agent_name (`str`): 호출할 ACP 에이전트의 이름입니다.
            arguments (dict[str, str] | str): 에이전트 호출에 전달된 인수입니다.
        """
        # 에이전트가 존재하는지 확인합니다.
        available_tools = {**self.tools}
        if agent_name not in available_tools:
            raise AgentToolExecutionError(
                f"Unknown agent {agent_name}, should be one of: {', '.join(available_tools)}.", self.logger
            )
        
        # 도구를 가져오고 인수에서 상태 변수를 대체합니다.
        tool = available_tools[agent_name]
        arguments = self._substitute_state_variables(arguments)
        
        try:
            # 적절한 인수로 에이전트를 호출합니다.
            if isinstance(arguments, dict):
                return await tool(**arguments, sanitize_inputs_outputs=True)
            elif isinstance(arguments, str):
                return await tool(arguments, sanitize_inputs_outputs=True)
            else:
                raise TypeError(f"Unsupported arguments type: {type(arguments)}")
                
        except TypeError as e:
            # 잘못된 인수를 처리합니다.
            description = getattr(tool, "description", "No description")
            error_msg = (
                f"Invalid call to agent '{agent_name}' with arguments {json.dumps(arguments)}: {e}\n"
                "You should call this agent with correct input arguments.\n"
                f"Expected inputs: {json.dumps(tool.inputs)}\n"
                f"Returns output type: {tool.output_type}\n"
                f"Agent description: '{description}'"
            )
            raise AgentToolCallError(error_msg, self.logger) from e
            
        except Exception as e:
            # 실행 오류를 처리합니다.
            error_msg = (
                f"Error executing agent '{agent_name}' with arguments {json.dumps(arguments)}: {type(e).__name__}: {e}\n"
                "Please try again or use another agent"
            )
            raise AgentToolExecutionError(error_msg, self.logger) from e

    async def run(self, query: str, max_steps: int = 10) -> str:
        """
        사용자 쿼리로 에이전트를 완료까지 실행합니다.
        
        Args:
            query (str): 사용자의 쿼리 또는 요청
            max_steps (int): 포기하기 전 최대 단계 수, 기본값 10
            
        Returns:
            str: 에이전트의 최종 답변
        """
        # LiteLLM에 맞는 형식으로 사용자 쿼리를 메모리에 초기화합니다.
        user_message = {"role": "user", "content": [{"type": "text", "text": query}]}
        system_message = {"role": "system", "content": [{"type": "text", "text": self.initialize_system_prompt()}]}
        self.input_messages = [system_message, user_message]
        
        # 최종 답변을 얻거나 최대 단계에 도달할 때까지 단계를 실행합니다.
        result = None
        for step_num in range(max_steps):
            self.logger.log(f"Step {step_num + 1}/{max_steps}", level=LogLevel.INFO)

            # 상태가 있는 경우 메모리 컨텍스트를 메시지에 추가합니다.
            if self.state and step_num > 0:
                memory_context = "Current memory state:\n"
                for key, value in self.state.items():
                    memory_context += f"- {key}: {value}\n"
                
                self.input_messages.append({
                    "role": "system",
                    "content": [{"type": "text", "text": memory_context}]
                })
            
            # 새로운 액션 단계를 생성하고 실행합니다.
            memory_step = ActionStep()
            
            try:
                result = await self.step(memory_step)
                
                # 최종 결과를 얻었다면 반환합니다.
                if result is not None:
                    return result
                
                # 그렇지 않으면, 다음 단계를 위해 관찰 결과를 입력 메시지에 추가합니다.
                if hasattr(memory_step, 'observations') and memory_step.observations:
                    # 보조 메시지가 있으면 추가합니다.
                    if hasattr(memory_step, 'model_output_message') and memory_step.model_output_message:
                        content = ""
                        if hasattr(memory_step.model_output_message, 'content'):
                            content = memory_step.model_output_message.content
                        elif hasattr(memory_step.model_output_message, 'raw'):
                            content = str(memory_step.model_output_message.raw)
                        
                        if content:
                            self.input_messages.append({
                                "role": "assistant",
                                "content": [{"type": "text", "text": content}]
                            })
                    
                    # 관찰 결과를 사용자 메시지로 추가합니다.
                    self.input_messages.append({
                        "role": "user", 
                        "content": [{"type": "text", "text": f"Observation: {memory_step.observations}"}]
                    })
            except Exception as e:
                self.logger.log(f"Error in step {step_num + 1}: {str(e)}", level=LogLevel.ERROR)
                # 대화에 오류 메시지를 추가합니다.
                self.input_messages.append({
                    "role": "user",
                    "content": [{"type": "text", "text": f"Error occurred: {str(e)}. Please try a different approach or provide a final answer."}]
                })
        
        # 최대 단계에 도달했지만 최종 답변이 없는 경우
        return "I wasn't able to complete this task within the maximum number of steps."

Overwriting fastacp.py


## 터미널에서 2개의 ACP 서버 재실행

```
uv run aidagent_server.py
uv run guideagent_server.py
```

## ACPCallingAgent 가져오기

In [4]:
%pip install -q smolagents

Note: you may need to restart the kernel to use updated packages.


In [5]:
import asyncio
import nest_asyncio

from acp_sdk.client import Client
from smolagents import LiteLLMModel

from fastacp import AgentCollection, ACPCallingAgent
from colorama import Fore

/opt/miniconda3/envs/lecture/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
print(ACPCallingAgent.__doc__)


    이 에이전트는 ToolCallingAgent가 로컬 도구를 사용하는 것과 유사하게,
    원격 ACP 에이전트를 대상으로 JSON과 유사한 ACP 에이전트 호출을 사용합니다.

    Args:
        acp_agents (`dict[str, Agent]`): 이 에이전트가 호출할 수 있는 ACP 에이전트들입니다.
        model (`Callable[[list[dict[str, str]]], ChatMessage]`): 에이전트의 행동을 생성할 모델입니다.
        prompt_templates ([`Dict[str, str]`], *optional*): 프롬프트 템플릿입니다.
        planning_interval (`int`, *optional*): 에이전트가 계획 단계를 실행할 간격입니다.
        **kwargs: 추가 키워드 인수입니다.
    


## 계층적 워크플로우 실행

In [7]:
nest_asyncio.apply()

In [16]:
model = LiteLLMModel(
    model_id="openai/gpt-4o-mini"
)

async def run_workflow() -> None:
    async with Client(base_url="http://localhost:8000") as first_aid, Client(base_url="http://localhost:8001") as guide_search:
        # 에이전트 발견
        agent_collection = await AgentCollection.from_acp(first_aid, guide_search)
        acp_agents = {agent.name: {'agent':agent, 'client':client} for client, agent in agent_collection.agents}
        print(acp_agents)
        
        # ACPCallingAgent에 에이전트들을 도구로 전달
        acpagent = ACPCallingAgent(acp_agents=acp_agents, model=model)
        
        # 사용자 질문으로 에이전트 실행
        result = await acpagent.run("증상별 응급처치에서 우선 처치 사항은 무엇인가?")
        print(Fore.YELLOW + f"Final result: {result}" + Fore.RESET)

In [17]:
asyncio.run(run_workflow())

{'policy_agent': {'agent': AgentManifest(name='policy_agent', description='이 에이전트는 학교 내에서 응급 환자 발생 시 응급 처치에 대한 질문을 처리하며, 응급처치 가이드 문서를 기반으로 답변을 찾기 위해 RAG 패턴을 사용한다.', metadata=Metadata(annotations=None, documentation=None, license=None, programming_language=None, natural_languages=None, framework=None, capabilities=None, domains=None, tags=None, created_at=None, updated_at=None, author=None, contributors=None, links=None, dependencies=None, recommended_models=None), input_content_types=['*/*'], output_content_types=['*/*']), 'client': <acp_sdk.client.client.Client object at 0x122d2d990>}, 'guide_agent': {'agent': AgentManifest(name='guide_agent', description='응급환자 발생시 관련 질문을 검색하여 지원하는 Agent다. 교내의 응급 발생 시 교직원과 학생이 응급 처치 방법을 검색하여 찾는 데 사용할 수 있다.', metadata=Metadata(annotations=None, documentation=None, license=None, programming_language=None, natural_languages=None, framework=None, capabilities=None, domains=None, tags=None, created_at=None, updated_at=None, author=None, contributors=None, 